# Scale benchmarking, payment reconciliation, and experiment packaging

**Goal.** Plan a bounded scale run, reconcile multi-rail payment projections, and package a complete reproducible experiment. This is the operational hand-off from a synthetic run to an auditable benchmark.

**Audience.** Data/ML scientists, fraud analysts, platform engineers, and researchers.

**Prerequisites.** Python 3.12+, a clean checkout, and the base FraudTwin install. The workflow is deterministic and runs offline; service integrations are deliberately out of scope here.

**Source size.** 1,000–10,000 logical payments. Every section writes only compact summaries, manifests, or fingerprints to a temporary directory.

**Interpretation.** Synthetic evidence demonstrates mechanics and invariants, not production prevalence or model performance guarantees.


## Scale and resource benchmarking



In [ ]:
from pathlib import Path

from fraudtwin.config import load_config
from fraudtwin.reproducibility import sha256_json
from fraudtwin.scale import require_scale_plan, shard_descriptors

config = load_config(Path("configs/scale-dev.yaml"))
plan = require_scale_plan(config, run_id="BENCH-SCALE-DEV")
shards = shard_descriptors(plan)
print(
    {
        "profile": plan.profile,
        "target_payments": plan.target_payments,
        "shards": plan.shard_count,
        "chunk_size": plan.chunk_size,
    }
)
assert plan.target_payments >= 1000

In [ ]:
rows = [{"shard_id": x.shard_id, "index": x.shard_index, "mapping": x.mapping} for x in shards]
print(rows)
assert [x["index"] for x in rows] == list(range(plan.shard_count))

In [ ]:
chunk_count = (plan.target_payments + plan.chunk_size - 1) // plan.chunk_size
print({"chunks": chunk_count, "checkpoint_frequency": plan.checkpoint_frequency_chunks})
assert chunk_count > 0

In [ ]:
profiles = {
    name: {"target": target, "chunks": (target + plan.chunk_size - 1) // plan.chunk_size}
    for name, target in (("dev", 1000), ("small", 100000), ("medium", 1000000))
}
print(profiles)

In [ ]:
checkpoint = {
    "run_id": plan.run_id,
    "configuration_hash": plan.configuration_hash,
    "completed_chunks": tuple(range(min(3, chunk_count))),
}
print(checkpoint)
assert max(checkpoint["completed_chunks"], default=-1) < chunk_count

In [ ]:
remaining = chunk_count - len(checkpoint["completed_chunks"])
print({"completed": len(checkpoint["completed_chunks"]), "remaining": remaining})
assert remaining >= 0

In [ ]:
logical_ids = tuple(f"PAY-{i:08d}" for i in range(plan.target_payments))
print({"logical_ids": len(logical_ids), "duplicates": len(logical_ids) - len(set(logical_ids))})
assert len(logical_ids) == len(set(logical_ids))

In [ ]:
manifest = {
    "profile": plan.profile,
    "target": plan.target_payments,
    "shards": plan.shard_count,
    "chunk_size": plan.chunk_size,
    "completed": checkpoint["completed_chunks"],
    "remaining": remaining,
}
manifest["fingerprint"] = sha256_json(manifest)
print(manifest)

In [ ]:
same = require_scale_plan(config, run_id="BENCH-SCALE-DEV")
print({"same_configuration": same.configuration_hash == plan.configuration_hash})
assert same.configuration_hash == plan.configuration_hash

In [ ]:
assert manifest["fingerprint"] == sha256_json(
    {k: v for k, v in manifest.items() if k != "fingerprint"}
)
print("The scale plan makes resume and reconciliation measurable before a large run is scheduled.")

## Multi-rail payment reconciliation



In [ ]:
from collections import Counter, defaultdict
from pathlib import Path

from fraudtwin.config import load_config
from fraudtwin.generation import generate
from fraudtwin.reproducibility import sha256_json

base = load_config(Path("configs/minimal.yaml"))
config = base.model_copy(
    update={"payments": base.payments.model_copy(update={"daily_target": 1000})}
)
data = generate(config, write=False)
payments = data.behavior.payments
events = data.behavior.payment_events
ledger = data.behavior.ledger_entries
print(
    {"run_id": data.run_id, "payments": len(payments), "events": len(events), "ledger": len(ledger)}
)
assert len(payments) >= 1000

In [ ]:
rail_counts = Counter(p.payment_rail for p in payments)
print(dict(rail_counts))
assert set(rail_counts) == {"CARD", "PIX", "ACCOUNT_TRANSFER"}

In [ ]:
event_counts = Counter(e.payment_rail for e in events)
print(dict(event_counts))
assert sum(event_counts.values()) == len(events)

In [ ]:
status_counts = Counter(p.current_status for p in payments)
print(dict(status_counts))
assert status_counts

In [ ]:
ledger_by_payment = defaultdict(float)
for e in ledger:
    ledger_by_payment[e.payment_id] += e.amount if e.entry_type == "CREDIT" else -e.amount
matched = sum(p.payment_id in ledger_by_payment for p in payments)
print({"payments_with_ledger": matched, "coverage": round(matched / len(payments), 3)})
assert matched > 0

In [ ]:
debit = round(sum(e.amount for e in ledger if e.entry_type == "DEBIT"), 2)
credit = round(sum(e.amount for e in ledger if e.entry_type == "CREDIT"), 2)
print({"debit_total": debit, "credit_total": credit, "difference": round(debit - credit, 2)})

In [ ]:
event_ids = {e.payment_id for e in events}
payment_ids = {p.payment_id for p in payments}
print(
    {
        "events_without_payment": len(event_ids - payment_ids),
        "payments_without_events": len(payment_ids - event_ids),
    }
)
assert event_ids <= payment_ids

In [ ]:
special = Counter(
    e.event_type
    for e in events
    if any(x in e.event_type for x in ("REFUND", "RETURN", "REVERSAL", "CHARGEBACK"))
)
print({"special_lifecycle_events": dict(special)})

In [ ]:
reconciliation = {
    "rail_counts": dict(rail_counts),
    "event_counts": dict(event_counts),
    "payments": len(payments),
    "ledger": len(ledger),
    "ledger_coverage": matched / len(payments),
}
reconciliation["fingerprint"] = sha256_json(reconciliation)
print(reconciliation)

In [ ]:
assert reconciliation["fingerprint"] == sha256_json(
    {k: v for k, v in reconciliation.items() if k != "fingerprint"}
)
assert reconciliation["ledger_coverage"] > 0
print("Lifecycle reconciliation uses domain IDs and does not rewrite source records.")

## Reproducible experiment packaging



In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

from fraudtwin import __version__
from fraudtwin.config import load_config
from fraudtwin.generation import generate
from fraudtwin.reproducibility import sha256_json

base = load_config(Path("configs/minimal.yaml"))
config = base.model_copy(
    update={"payments": base.payments.model_copy(update={"daily_target": 1000})}
)
data = generate(config, write=False)
counts = {
    "payments": len(data.behavior.payments),
    "events": len(data.behavior.payment_events),
    "ledger": len(data.behavior.ledger_entries),
    "dataset": data.require_dataset().frame.height,
}
print({"version": __version__, "run_id": data.run_id, **counts})
assert counts["payments"] >= 1000

In [ ]:
schema_versions = {
    "events": sorted({e.schema_version for e in data.behavior.payment_events}),
    "manifest": data.manifest.schema_versions,
}
print(schema_versions)
assert schema_versions["events"]

In [ ]:
identity = {
    "run_id": data.run_id,
    "seed": config.simulation.seed,
    "configuration_hash": data.manifest.scenario_config_hash,
    "counts": counts,
    "schema_versions": schema_versions,
}
identity["data_fingerprint"] = sha256_json(
    {
        "payments": tuple(p.payment_id for p in data.behavior.payments),
        "events": tuple(e.event_id for e in data.behavior.payment_events),
    }
)
print(identity)

In [ ]:
experiment = {
    "package": "fraudtwin",
    "version": __version__,
    "identity": identity,
    "purpose": "bounded reproducibility",
    "limitations": "synthetic data; local timing only",
}
experiment["fingerprint"] = sha256_json(experiment)
print({"experiment_fingerprint": experiment["fingerprint"]})

In [ ]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "experiment.json"
    path.write_text(json.dumps(experiment, sort_keys=True, indent=2, default=str), encoding="utf-8")
    loaded = json.loads(path.read_text(encoding="utf-8"))
print({"artifact": path.name, "keys": sorted(loaded)})
assert loaded["fingerprint"] == experiment["fingerprint"]

In [ ]:
repeat = generate(config, write=False)
print(
    {
        "same_run_id": repeat.run_id == data.run_id,
        "same_events": tuple(e.event_id for e in repeat.behavior.payment_events)
        == tuple(e.event_id for e in data.behavior.payment_events),
    }
)
assert repeat.run_id == data.run_id

In [ ]:
release = {
    "experiment_fingerprint": experiment["fingerprint"],
    "configuration_hash": data.manifest.scenario_config_hash,
    "seed": config.simulation.seed,
    "counts": counts,
}
print(release)

In [ ]:
release["fingerprint"] = sha256_json(release)
print(release)
assert release["fingerprint"]

In [ ]:
print({"reproducibility_fields": sorted(experiment["identity"])})

In [ ]:
assert release["fingerprint"] == sha256_json(
    {k: v for k, v in release.items() if k != "fingerprint"}
)
print(
    "Configuration, seed, package version, schema, counts, and fingerprints are ready to archive."
)

## Verification and next step

Re-run the offline cells from a clean checkout and compare the printed fingerprints. For service-backed publication, continue with the relevant integration runbook after this notebook; do not treat synthetic metrics as a deployment SLO.
